# Phase 2: Gemma 4 E4B — Text-Only FIM Fine-Tuning (QLoRA)

Train a tab-completion model on Gemma 4 E4B using Fill-in-the-Middle (FIM)
with QLoRA on Kaggle T4 GPU (16 GB VRAM).

Uses standard HuggingFace PEFT + bitsandbytes (no unsloth — avoids version conflicts).

**Papers referenced:**
- [AST-FIM] Structure-Aware FIM Pretraining (arXiv:2506.00204)
- [HLP] Horizon-Length Prediction (EMNLP 2025)
- [Mellum] Production-Grade Code Completion (arXiv:2510.05788)
- [Curriculum-FIM] Curriculum-Based FIM (arXiv:2412.16589)

**Requirements:** Kaggle notebook with **GPU T4 x2** accelerator (Settings → Accelerator).
P100 is NOT supported by PyTorch 2.10+ (sm_60 < minimum sm_70).

In [1]:
# Cell 1: Verify environment and install deps
import torch

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"
print(f"PyTorch {torch.__version__}, CUDA {torch.version.cuda}, GPU: {gpu_name}")

cap = torch.cuda.get_device_capability(0)
assert cap[0] >= 7, f"GPU sm_{cap[0]}{cap[1]} not supported — switch to T4 (sm_75) in Settings → Accelerator"

# Upgrade transformers — Kaggle's pre-installed version may not support Gemma 4
!pip install -q --upgrade transformers 2>&1 | tail -3
!pip install -q peft bitsandbytes accelerate trl datasets pyyaml tqdm 2>&1 | tail -3

import transformers, peft, bitsandbytes
print(f"transformers {transformers.__version__}, peft {peft.__version__}, bnb {bitsandbytes.__version__}")

from transformers import AutoConfig
try:
    _ = AutoConfig.from_pretrained("google/gemma-4-E4B", trust_remote_code=True)
    print("Gemma 4 config loads OK")
except Exception as e:
    print(f"Gemma 4 not in this transformers release — installing from source...")
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/huggingface/transformers.git"])
    print("Restart kernel and re-run this cell.")

print("Setup complete.")

PyTorch 2.0.1+cpu, CUDA None, GPU: NONE


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
# Cell 2: Gemma 4 compatibility note
# Do NOT patch Gemma4ClippableLinear.__bases__ on recent transformers builds.
# That patch can break model construction (Linear.__init__ arg mismatch).
# Instead, we keep classes unchanged and target the inner *.linear modules for LoRA.
print("Skipping Gemma4ClippableLinear base-class patch (safe mode)")

In [ ]:
# Cell 3: Load Gemma 4 with 4-bit quantization (force 2-GPU sharding)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import transformers.modeling_utils as modeling_utils
import torch, gc, os

# Prefer E2B for Phase 2 stability on Kaggle T4.
MODEL_CANDIDATES = ["google/gemma-4-E2B"]
MAX_SEQ_LENGTH = 256

# Reduce allocator fragmentation and skip warmup allocations.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
modeling_utils.caching_allocator_warmup = lambda *args, **kwargs: None

def _to_cuda_index(dev):
    if isinstance(dev, int):
        return dev
    if isinstance(dev, str) and dev.startswith("cuda:"):
        try:
            return int(dev.split(":", 1)[1])
        except Exception:
            return None
    if isinstance(dev, torch.device) and dev.type == "cuda":
        return 0 if dev.index is None else dev.index
    return None

def _used_cuda_devices(m):
    dmap = getattr(m, "hf_device_map", None)
    used = set()

    if isinstance(dmap, dict):
        for v in dmap.values():
            idx = _to_cuda_index(v)
            if idx is not None:
                used.add(idx)

    # Fallback for paths without hf_device_map metadata
    if not used:
        for _, p in m.named_parameters():
            if p.device.type == "cuda":
                used.add(0 if p.device.index is None else p.device.index)
                if len(used) >= 2:
                    break

    return sorted(used)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

num_gpus = torch.cuda.device_count()
assert num_gpus >= 2, "Enable 2 GPUs in Kaggle settings (T4 x2)."
for i in range(num_gpus):
    print(f"GPU{i}: {torch.cuda.get_device_name(i)}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Try several placement strategies to maximize 2-GPU usage.
placement_attempts = [
    ("balanced", {0: "5GiB", 1: "14GiB"}),
    ("balanced", {0: "4GiB", 1: "14GiB"}),
    # Sequential can force at least some modules onto GPU0 when balanced keeps everything on GPU1.
    ("sequential", {0: "2GiB", 1: "14GiB"}),
    ("sequential", {0: "3GiB", 1: "14GiB"}),
]

model = None
tokenizer = None
MODEL_NAME = None
last_error = None
best_single_gpu_model = None
best_single_gpu_tokenizer = None
best_single_gpu_name = None

for candidate in MODEL_CANDIDATES:
    print(f"\nTrying model: {candidate}")
    tok = AutoTokenizer.from_pretrained(candidate, trust_remote_code=True)

    for device_map_mode, max_memory in placement_attempts:
        try:
            print(f"  Trying device_map={device_map_mode}, max_memory={max_memory}")
            mdl = AutoModelForCausalLM.from_pretrained(
                candidate,
                quantization_config=bnb_config,
                device_map=device_map_mode,
                max_memory=max_memory,
                dtype=torch.float16,
                attn_implementation="sdpa",
                trust_remote_code=True,
                low_cpu_mem_usage=True,
            )

            used = _used_cuda_devices(mdl)
            print(f"  Used GPUs: {used}")

            if len(used) >= 2:
                tokenizer = tok
                model = mdl
                MODEL_NAME = candidate
                break

            # Keep best single-GPU fallback instead of failing hard.
            if best_single_gpu_model is None:
                best_single_gpu_model = mdl
                best_single_gpu_tokenizer = tok
                best_single_gpu_name = candidate
            else:
                del mdl
                gc.collect()
                torch.cuda.empty_cache()

        except Exception as e:
            last_error = e
            print(f"  Failed placement: {type(e).__name__}: {e}")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if model is not None:
        break

if model is None and best_single_gpu_model is not None:
    print("WARNING: Could not shard across both GPUs; continuing with single-GPU placement.")
    tokenizer = best_single_gpu_tokenizer
    model = best_single_gpu_model
    MODEL_NAME = best_single_gpu_name

if model is None:
    raise RuntimeError(
        "Could not load model with any placement strategy. "
        f"Last error: {type(last_error).__name__}: {last_error}"
    )

model.config.use_cache = False

print(f"Model loaded: {MODEL_NAME}")
print(f"Vocab size: {len(tokenizer)}")

used = _used_cuda_devices(model)
print(f"Final used GPUs: {used}")
for dev_idx in used:
    props = torch.cuda.get_device_properties(dev_idx)
    print(f"GPU{dev_idx} memory: {torch.cuda.memory_allocated(dev_idx)/1e9:.1f} GB / {props.total_memory/1e9:.1f} GB")

In [ ]:
# Cell 4: Configure FIM markers and attach LoRA adapters (OOM-safe + robust target matching)
from peft import LoraConfig, get_peft_model

FIM_PREFIX = "<fim_prefix>"
FIM_SUFFIX = "<fim_suffix>"
FIM_MIDDLE = "<fim_middle>"

# IMPORTANT: Do NOT add special tokens for E4B on T4.
# Resizing embeddings triggers a large temporary allocation and OOM.
print("Skipping tokenizer.add_special_tokens / resize_token_embeddings to avoid OOM")

# Freeze base model first.
for p in model.parameters():
    p.requires_grad = False

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

# Auto-detect valid inner linear projection modules across Gemma4 variants.
# IMPORTANT: target ONLY `*.linear` children; `Gemma4ClippableLinear` wrappers are unsupported by PEFT.
wanted_suffixes = {
    "q_proj.linear", "k_proj.linear", "v_proj.linear", "o_proj.linear",
    "gate_proj.linear", "up_proj.linear", "down_proj.linear",
}
target_modules = []
for n, m in model.named_modules():
    if not any(n.endswith(s) for s in wanted_suffixes):
        continue
    cls = m.__class__.__name__.lower()
    # Accept torch Linear and bitsandbytes Linear4bit/Linear8bit modules.
    if "linear" in cls:
        target_modules.append(n.split(".")[-2] + ".linear")

target_modules = sorted(set(target_modules))
if not target_modules:
    raise RuntimeError("No supported inner linear modules found for LoRA injection.")

print(f"LoRA target modules: {target_modules}")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)")
if trainable == 0:
    raise RuntimeError("LoRA attached zero trainable parameters; aborting before trainer.")

In [ ]:
# Cell 5: Load FIM dataset
# Priority:
# 1) Use uploaded Phase 1 dataset from Kaggle input (recommended)
# 2) Fallback to open code_search_net and build simple FIM on-the-fly

from datasets import load_from_disk, load_dataset, Dataset
from pathlib import Path
import random

# Make this cell standalone-safe if Cell 4 has not run yet.
FIM_PREFIX = globals().get("FIM_PREFIX", "<fim_prefix>")
FIM_SUFFIX = globals().get("FIM_SUFFIX", "<fim_suffix>")
FIM_MIDDLE = globals().get("FIM_MIDDLE", "<fim_middle>")

candidate_paths = [
    "/kaggle/input/gemma4-fim-dataset",                 # if uploaded as raw HF folder
    "/kaggle/input/gemma4-fim-dataset/fim_dataset",     # if wrapped inside outer folder
    "/kaggle/input/fim-dataset",
    "/kaggle/input/gemma4-fim",
]

train_dataset = None
for p in candidate_paths:
    if Path(p).exists():
        try:
            ds = load_from_disk(p)
            if isinstance(ds, dict) or hasattr(ds, "keys"):
                train_dataset = ds["train"] if "train" in ds else None
            else:
                train_dataset = ds
            if train_dataset is not None:
                print(f"Loaded pre-processed FIM dataset from: {p}")
                print(f"Train samples: {len(train_dataset)}")
                break
        except Exception as e:
            print(f"Found {p} but could not load_from_disk: {e}")

if train_dataset is None:
    print("Pre-processed FIM dataset not found. Falling back to open code_search_net...")

    raw = load_dataset("code_search_net", "python", split="train", streaming=True)

    MAX_SAMPLES = 50_000
    samples = []
    for i, item in enumerate(raw):
        if i >= MAX_SAMPLES:
            break

        content = item.get("whole_func_string", "")
        if len(content) < 50 or len(content) > 50_000:
            continue

        # Simple random FIM fallback
        if random.random() < 0.5:
            samples.append({"text": content})
        else:
            start = random.randint(1, max(1, len(content) - 2))
            end = random.randint(start, min(start + 500, len(content)))
            prefix = content[:start]
            middle = content[start:end]
            suffix = content[end:]

            if random.random() < 0.5:
                fim_text = f"{FIM_PREFIX}{prefix}{FIM_SUFFIX}{suffix}{FIM_MIDDLE}{middle}"
            else:
                fim_text = f"{FIM_SUFFIX}{suffix}{FIM_PREFIX}{prefix}{FIM_MIDDLE}{middle}"
            samples.append({"text": fim_text})

    train_dataset = Dataset.from_list(samples)
    print(f"Created {len(train_dataset)} samples from code_search_net")

# Keep Phase 2 compute bounded on free GPUs
MAX_TRAIN_SAMPLES = 25000
if len(train_dataset) > MAX_TRAIN_SAMPLES:
    train_dataset = train_dataset.select(range(MAX_TRAIN_SAMPLES))
print(f"Using {len(train_dataset)} samples for Phase 2")

In [ ]:
# Cell 6: Train (hard OOM-safe profile)
from trl import SFTTrainer
from transformers import TrainingArguments
import inspect, torch

OUTPUT_DIR = "/kaggle/working/gemma4-e4b-fim"

# Enforce a bounded dataset in this cell (even if Cell 5 wasn't re-run)
MAX_TRAIN_SAMPLES = 5000
MAX_TEXT_CHARS = 400
if len(train_dataset) > MAX_TRAIN_SAMPLES:
    train_dataset = train_dataset.select(range(MAX_TRAIN_SAMPLES))
train_dataset = train_dataset.map(
    lambda ex: {"text": ex["text"][:MAX_TEXT_CHARS]},
    desc="Hard clipping text length",
)

# Keep sequence length local to this cell so stale globals don't hurt us.
TRAIN_MAX_SEQ_LENGTH = 128

train_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=1e-4,
    fp16=False,
    bf16=False,
    warmup_steps=30,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    report_to="none",
    gradient_checkpointing=True,
    optim="adamw_torch",
)

# Prevent Trainer from wrapping with torch.nn.DataParallel on multi-GPU.
model.is_parallelizable = True
model.model_parallel = True

sig = inspect.signature(SFTTrainer.__init__).parameters
trainer_kwargs = {
    "model": model,
    "train_dataset": train_dataset,
    "args": train_args,
}

if "tokenizer" in sig:
    trainer_kwargs["tokenizer"] = tokenizer
elif "processing_class" in sig:
    trainer_kwargs["processing_class"] = tokenizer

if "dataset_text_field" in sig:
    trainer_kwargs["dataset_text_field"] = "text"
if "max_seq_length" in sig:
    trainer_kwargs["max_seq_length"] = TRAIN_MAX_SEQ_LENGTH
elif "max_length" in sig:
    trainer_kwargs["max_length"] = TRAIN_MAX_SEQ_LENGTH
if "packing" in sig:
    trainer_kwargs["packing"] = False

trainer = SFTTrainer(**trainer_kwargs)
# Keep Trainer single-process while model itself is sharded by accelerate.
trainer.args._n_gpu = 1

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {n_trainable:,}")
assert n_trainable > 0, "No trainable params found after LoRA injection"

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Training on {len(train_dataset)} samples | max_seq={TRAIN_MAX_SEQ_LENGTH} | max_chars={MAX_TEXT_CHARS}")
trainer.train()
print("Training complete!")

In [ ]:
# Cell 7: Save the fine-tuned model
SAVE_PATH = "/kaggle/working/gemma4-e4b-fim-final"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Model saved to {SAVE_PATH}")

In [ ]:
# Cell 8: Quick sanity check — test FIM completion
model.eval()

test_prompt = f"""{FIM_PREFIX}def fibonacci(n):
    if n <= 1:
        return n
{FIM_SUFFIX}
    return result
{FIM_MIDDLE}"""

model_device = next(model.parameters()).device
inputs = tokenizer(test_prompt, return_tensors="pt").to(model_device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.2,
        do_sample=True,
    )

decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)
print("=" * 50)
print("FIM Completion Test:")
print("=" * 50)
print(decoded)

In [ ]:
# Cell 9: Merge LoRA weights and export
# GGUF conversion is done locally after download (llama.cpp's convert tool).
# Here we save the merged full-precision model for download.

model_tag = MODEL_NAME.split("/")[-1].lower()
MERGED_PATH = f"/kaggle/working/{model_tag}-fim-merged"

merged_model = model.merge_and_unload()
merged_model.save_pretrained(MERGED_PATH)
tokenizer.save_pretrained(MERGED_PATH)

print(f"Merged model saved to {MERGED_PATH}")
print("Download, then convert to GGUF locally:")
print(f"  python llama.cpp/convert_hf_to_gguf.py {model_tag}-fim-merged --outtype q4_k_m")

In [ ]:
# Cell 10: Zip + download link for final output only (with progress bar)
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from tqdm.auto import tqdm
from IPython.display import FileLink, display

# Prefer merged output as the final artifact; fall back to final adapter folder.
candidates = [
    Path("/kaggle/working/gemma-4-e2b-fim-merged"),
    Path("/kaggle/working/gemma-4-e4b-fim-merged"),
    Path("/kaggle/working/gemma4-e4b-fim-final"),
]

target = next((p for p in candidates if p.exists()), None)
if target is None:
    raise FileNotFoundError(
        "No final output folder found. Expected one of: " + ", ".join(str(p) for p in candidates)
    )

zip_path = Path("/kaggle/working") / f"{target.name}.zip"
if zip_path.exists():
    zip_path.unlink()

files = [p for p in target.rglob("*") if p.is_file()]
total_bytes = sum(p.stat().st_size for p in files)

print("Final output folder:", target)
print(f"Files to zip: {len(files)} | Total size: {total_bytes / (1024**3):.2f} GiB")

with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as zf:
    with tqdm(total=total_bytes, unit="B", unit_scale=True, desc="Zipping") as pbar:
        for f in files:
            arcname = f.relative_to(target)
            zf.write(f, arcname=arcname)
            pbar.update(f.stat().st_size)

print("Created zip:", zip_path)
print("\nDownload link:")
display(FileLink(str(zip_path)))